In [6]:
import csv
import asyncio
import os
from datetime import datetime
from dotenv import load_dotenv

# Load config from .env
load_dotenv()
PASS_MARK = int(os.getenv("PASS_MARK", 50))
LATE_FINE = int(os.getenv("LATE_FINE_PER_DAY", 10))

# Custom exception for invalid records
class RecordError(Exception):
    pass

# Decorator for logging errors
def log_errors(func):
    def wrapper(*args, **kwargs):
        try:
            return func(*args, **kwargs)
        except RecordError as e:
            with open("error_log.txt", "a") as f:
                f.write(str(e) + "\n")
            print("Error:", e)
    return wrapper

# Student class using OOP
class Student:
    def __init__(self, sid, name, marks, days_late):
        self.sid = sid
        self.name = name
        self.marks = int(marks)
        self.days_late = int(days_late)

    @property
    def status(self):
        return "PASS" if self.marks >= PASS_MARK else "FAIL"

    @property
    def fine(self):
        return self.days_late * LATE_FINE

    def __str__(self):
        return f"{self.sid} | {self.name} | Marks:{self.marks} | {self.status} | Fine:{self.fine}"

# Main portal class
class AcademicPortal:
    def __init__(self):
        self.students = []

    @log_errors
    def validate_record(self, row):
        if len(row) != 4:
            raise RecordError(f"Invalid row length: {row}")
        sid, name, marks, days = row
        if not marks.isdigit():
            raise RecordError(f"Invalid marks for {name}: {marks}")
        return Student(sid, name, marks, days)

    async def load_file(self, filename):
        await asyncio.sleep(0.5)  # simulate delay
        with open(filename) as f:
            reader = csv.reader(f)
            next(reader)  # skip header
            for row in reader:
                student = self.validate_record(row)
                if student:
                    self.students.append(student)

    def show_summary(self):
        total = len(self.students)
        failed = list(filter(lambda s: s.status == "FAIL", self.students))
        print("\nTOTAL STUDENTS:", total)
        print("FAILED STUDENTS:", len(failed))

    def generate_report(self):
        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_name = f"report_{now}.txt"
        with open(report_name, "w") as f:
            f.write("STUDENT REPORT\n\n")
            for s in self.students:
                f.write(str(s) + "\n")
        print("\nReport generated:", report_name)

# Function to create sample CSV files
def create_sample_csv():
    data1 = [["id","name","marks","late_days"],["1","Ali","78","2"],["2","Sara","45","0"]]
    data2 = [["id","name","marks","late_days"],["3","John","88","1"],["4","Emma","30","3"]]
    data3 = [["id","name","marks","late_days"],["5","David","67","0"],["6","Anna","40","2"]]
    files = [("students1.csv", data1), ("students2.csv", data2), ("students3.csv", data3)]
    for filename, data in files:
        with open(filename,"w",newline="") as f:
            writer = csv.writer(f)
            writer.writerows(data)

# Main async function
async def main():
    create_sample_csv()
    portal = AcademicPortal()
    await asyncio.gather(
        portal.load_file("students1.csv"),
        portal.load_file("students2.csv"),
        portal.load_file("students3.csv")
    )
    portal.show_summary()
    print("\nSTUDENT RECORDS\n")
    for s in portal.students:
        print(s)
    portal.generate_report()

await main()


TOTAL STUDENTS: 6
FAILED STUDENTS: 3

STUDENT RECORDS

1 | Ali | Marks:78 | PASS | Fine:20
2 | Sara | Marks:45 | FAIL | Fine:0
3 | John | Marks:88 | PASS | Fine:10
4 | Emma | Marks:30 | FAIL | Fine:30
5 | David | Marks:67 | PASS | Fine:0
6 | Anna | Marks:40 | FAIL | Fine:20

Report generated: report_20260308_231044.txt
